# IDAT calling and conversion to Plink1.9 and Plink2.0

This notebook converts .idat files from selected samples into .ped/.map format and calls all samples together generating Plink1.9 and Plink2.0 format output.

## Import packages

List of packages used in this notebook:

* subprocess
* concurrent.futures (ThreadPoolExecutor)

In [ ]:
# System and path
import os
import sys
import glob
import shutil
import gzip
import bz2
from pathlib import Path
from datetime import date

# Process and parallelization
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import subprocess
import requests
import multiprocessing
import logging
from collections import Counter, defaultdict
from itertools import combinations, product
from functools import reduce

# Dataframes and other formats
import numpy as np
import pandas as pd
import polars as pl
import openpyxl
import re
import json
import csv
import pickle

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

## Variables and Paths

### Tools

List of tools used in this notebook:

* iaap-cli
* Plink1.9
* Plink2.0

In [ ]:
# Hestia NGS Software
tools = f"/data/hestia/NGS_Software"

# Plink1.9 and Plink2.0 path
plink1 = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

### Paths

List of paths used in this notebook:

Input
* /path/to/idat_directory
* /path/to/manifest_file
* /path/to/cluster_file
* /path/to/master_file  

Output
* /path/to/pedmap_directory
* /path/to/plink_directory
* /path/to/vcf_directory

In [ ]:
# Directories

# Home dir
home = Path("/data/hestia/misayan")

# Hestia NGS Software
tools = Path(f"/data/hestia/NGS_Software")

# Main directory
main_dir = Path(f"{home}/GWAS")
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = Path(f"{main_dir}/DATA")
DATA_DIR = data_dir     ### alias
os.makedirs(data_dir, exist_ok=True)

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = Path(f"{data_dir}/META")
META_DIR = meta_dir     ### alias
os.makedirs(meta_dir, exist_ok=True)

## Generate a manifest file

In [ ]:
key = pd.read_excel(f'{home}/manifest.xlsx', engine='openpyxl')
print(key.shape[0])
key.head()

In [ ]:
# Select columns to use
old_cols = ['GP2sampleID', 'SentrixBarcode_A', 'SentrixPosition_A', 'biological_sex_for_qc', 'clinical_id']

# Rename columns
new_cols = ['Sample_ID', 'SentrixBarcode_A', 'SentrixPosition_A', 'Gender', 'Sample_Name']

In [ ]:
# Create a subset of columns
sheet = key[old_cols].copy()

# Rename columns
sheet.columns = new_cols

# Add IDs
sheet['ID'] = sheet['SentrixBarcode_A'].astype(str) + '_' + sheet['SentrixPosition_A'].astype(str)

# Number of samples
print(sheet.shape[0])
sheet.head()

In [ ]:
# Read all_idats.txt that contains all paths to idats
# Obrain by running
# find /path/to/main/dir -iname "*_Grn.idat"
df = pd.read_csv(f'{home}/CAT-PD/DATA/all_idats.txt', sep='\t', header=None, names=['Path'])

# Split path to get Sentrix Barcode
df['ID'] = df['Path'].str.split('/').str[-1]
df = df[~df['ID'].str.contains('._20')]

# Drop potential duplicates
df = df.drop_duplicates(subset='ID', keep='first')

# Remove idat extension
df['ID'] = df['ID'].str.replace('_Grn.idat', '', regex=False)

# Remove filename from Path
df['Path'] = df['Path'].str.rsplit('/', n=1).str[0]

df.head()

In [ ]:
key_paths = pd.merge(sheet, df, on='ID', how='left')
key_paths = key_paths.drop(columns=['ID'])
key_paths

In [ ]:
key_paths.to_csv(f'{data_dir}/manifest_for_GenomeStudio.csv', sep=',', index=False)

In [ ]:
with open(f'{data_dir}/manifest_for_GenomeStudio.csv', "w") as f:
    f.write("[Data]\n")
    key_paths.to_csv(f, sep=",", index=False)

#### Chunk the manifest file

In [ ]:
# Output pedmap
tmp_pedmap_path = Path(f"{data_dir}/tmp")
os.makedirs(tmp_pedmap_path, exist_ok=True)
plink_output_path = Path(f"{data_dir}/PLINK")
os.makedirs(plink_output_path, exist_ok=True)

In [ ]:
key_paths = pd.read_csv(f'{data_dir}/manifest_for_GenomeStudio.csv', sep=',', header=0, comment='[')
key_paths.head()

#### Group by SentrixBarcode_A

In [ ]:
for i, (barcode, df_group) in enumerate(key_paths.groupby('SentrixBarcode_A'), 1):
    out = tmp_pedmap_path / f"manifest_for_GenomeStudio_{barcode}.csv"
    with out.open("w") as f:
        f.write("[Data]\n")
        df_group.to_csv(f, sep=",", index=False)

## IDAT calling

In [ ]:
# Cluster file
cluster = f"{home}/Neurobooster/recluster_09272022.egt"

# Manifest file
manifest = f"{home}/Neurobooster/NeuroBooster_20042459_A2.bpm"

# Master file
master_key = f'{data_dir}/manifest_for_GenomeStudio.csv'

# Number of threads (default 1 for parallelization)
threads = 1

In [ ]:
def call_idats(manifest_master=None, output=None, cluster=None, manifest=None, threads=str(1)):
    runGenomeStudio = [
        "iaap-cli", "gencall",
        manifest,
        cluster,
        output,
        "--sample-sheet", manifest_master,
        "--num-threads", str(threads),
        "--gencall-cutoff", str(0.15),
        "--output-ped",
    ]
    subprocess.run(runGenomeStudio)

Manifest structure for each barcode:

```bash
[Data]
Sample_ID,SentrixBarcode_A,SentrixPosition_A,Gender,Sample_Name,Path
```

In [ ]:
# Prepare jobs
jobs = []
for i, (barcode, df_group) in enumerate(key_paths.groupby('SentrixBarcode_A'), 1):
        manifest_file = Path(tmp_pedmap_path) / f"manifest_for_GenomeStudio_{barcode}.csv"
        jobs.append(dict(
            manifest_master=manifest_file,
            output=tmp_pedmap_path,
            cluster=cluster,
            manifest=manifest,
            threads="1"
        ))

In [ ]:
# Function to run jobs for ProcessPoolExecutor
def run_job(kw):
    call_idats(**kw)
    return True

In [ ]:
# Run with ProcessPoolExecutor in parallel
with ProcessPoolExecutor(max_workers=64) as exe:
    list(exe.map(run_job, jobs))

#### Make a list of ped and map files for merging

In [ ]:
# Define paths
# map_file = Path(f'{meta_dir}/gwas.pedmap')
map_file = Path(tmp_pedmap_path) / "pedmap_merge_list.txt"
map_name = Path(tmp_pedmap_path) / "NeuroBooster_20042459_A2.map"

# Create pedmap file for merging
with map_file.open("w") as f:
    for ped in sorted(Path(tmp_pedmap_path).glob("20*.ped")):
        f.write(f"{ped.name}\t{map_name}\n")

In [ ]:
pairs = []
with map_file.open() as f:
    for line in f:
        p, m = line.rstrip().split("\t")
        pairs.append((p, m))

In [ ]:
# Define chunks
chunks = [[] for _ in range(16)]

# Subset pairs
for i, (p, m) in enumerate(pairs):
    chunks[i % 16].append(f"{p}\t{m}")

In [ ]:
# Initialize empty output files list
out_files = []

# Save chunks into separate files
for i, lines in enumerate(chunks):
    if lines:
        out = Path(meta_dir) / f"pedmap_chunk_{i+1}.txt"
        out.write_text("\n".join(lines))
        out_files.append(out)

In [ ]:
# Function to merge pedmap files into chunks
def exportPedmap(filename, output=None, threads=1):
    if output is None:
        output = filename.stem
    subprocess.run(
        ["plink", 
         "--merge-list", Path(meta_dir) / str(filename), 
         "--threads", str(threads), 
         "--export", "ped", 
         "--out", Path(meta_dir) / output],
        check=True)

<div class="alert alert-block alert-info">
<b>Tip:</b> Run with either ThreadPoolExecutor or ProcessPoolExecutor.
</div>

In [ ]:
if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=16) as ex:
        ex.map(exportPedmap, out_files)

## Call all pedmap together

In [ ]:
# Define merge list with pedmap chunks
# out = Path(tmp_pedmap_path) / "pedmap_all.txt"
# out = Path(f'{meta_dir}/pedmap_to_merge.tsv')

# Get all ped and corresponding map
with out.open("w") as w:
    for f in sorted(Path(meta_dir).glob("pedmap_chunk_*.ped")):
        ped = f
        mapf = f.with_suffix(".map")
        w.write(f"{ped}\t{mapf}\n")

In [ ]:
exportPedmap(out, f"{data_dir}/RAW/CATPD_R11_GWAS", str(16))

In [ ]:
# Define function to merge pedmap files into chunks
def getPfile(filename, output=None, threads=1):
    if output is None:
        output = filename.stem
    subprocess.run(
        ["plink2", 
         "--pedmap", Path(meta_dir) / str(filename), 
         "--threads", str(threads), 
         "--make-pgen",
         "--out", Path(meta_dir) / output],
        check=True)
    
getPfile(f"{data_dir}/RAW/CATPD_R11_GWAS", f"{data_dir}/RAW/CATPD_R11_GWAS", threads=str(16))